# Fine-Tuning Whisper for ASR

Whisper's pretrained weights work well for widely-spoken languages with lots of training data, but performance degrades significantly for low-resource languages, domain-specific vocabulary, or strong accents. Fine-tuning adapts a pretrained Whisper model to your target language and domain using a relatively small amount of labeled data. This notebook walks through every step: dataset preparation, preprocessing, data augmentation, and running a fine-tuning loop with HuggingFace's `Seq2SeqTrainer`.

In [ ]:
# pip install transformers datasets torchaudio librosa soundfile evaluate jiwer  # uncomment if needed
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
import librosa
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from datasets import load_dataset, Audio
import evaluate

print(f"torch: {torch.__version__}")
print(f"torchaudio: {torchaudio.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 1. Preparing an Audio Dataset

### The Common Voice Dataset

Mozilla Common Voice is a crowdsourced multilingual speech dataset. Version 11.0 covers 100+ languages. We'll use it here as our fine-tuning source.

To load it, you need a HuggingFace account and must accept the dataset terms of use at [huggingface.co/datasets/mozilla-foundation/common_voice_11_0](https://huggingface.co/datasets/mozilla-foundation/common_voice_11_0). After that, run `huggingface-cli login` in your terminal once.

Below, we use `"cy"` (Welsh) as the target language because it has a manageable dataset size and is a genuine low-resource language where fine-tuning provides clear gains over the base model. Replace with any language code from the dataset page.

### Loading Common Voice 13.0 French

The cell below loads the French split from Common Voice 13.0 and prints the full dataset schema. You need a HuggingFace account and must accept the dataset terms at `huggingface.co/datasets/mozilla-foundation/common_voice_13_0`. Run `huggingface-cli login` once in your terminal first.

In [ ]:
# pip install datasets transformers soundfile  # uncomment if needed
from datasets import load_dataset, Audio

# Load French train + validation splits from Common Voice 13.0
# Using streaming=False downloads the data; set streaming=True for large splits
cv_fr = load_dataset(
    "mozilla-foundation/common_voice_13_0",
    "fr",
    split="train+validation",
    trust_remote_code=False,
)

print("=== Dataset overview ===")
print(cv_fr)
print()
print("=== Column names ===")
print(cv_fr.column_names)
print()
print("=== Features schema ===")
for col, feat in cv_fr.features.items():
    print(f"  {col:<20} {feat}")
print()
print("=== First example (raw) ===")
ex = cv_fr[0]
for k, v in ex.items():
    if k == "audio":
        print(f"  audio: dict with keys {list(v.keys())}")
        print(f"    path:         {v['path']}")
        print(f"    sampling_rate:{v['sampling_rate']} Hz")
        print(f"    array shape:  {len(v['array'])} samples "
              f"({len(v['array'])/v['sampling_rate']:.2f}s)")
    else:
        val = str(v)
        print(f"  {k:<20} {val[:80]}")

In [ ]:
LANGUAGE = "cy"          # Welsh; change to e.g. "fr", "de", "sw", "ta" as needed
LANGUAGE_NAME = "welsh"  # Used in Whisper's forced-language token
MODEL_ID = "openai/whisper-tiny"  # tiny for fast iteration; use small/medium for production

# Load only the train and test splits, streaming the data rather than downloading everything
# trust_remote_code=False is the safe default
common_voice = load_dataset(
    "mozilla-foundation/common_voice_11_0",
    LANGUAGE,
    split={"train": "train", "test": "test"},
    trust_remote_code=False,
)

print(common_voice)
print("\nFeature names:", common_voice["train"].column_names)
print("\nFirst example:")
print(common_voice["train"][0])

### Resampling to 16 kHz

Common Voice audio is stored at 48 kHz. Whisper requires 16 kHz. The HuggingFace `Audio` feature handles this automatically when you call `.cast_column()`, which is the cleanest approach when working with `datasets`.

We also remove columns we won't use to reduce memory overhead.

### Audio Preprocessing: 16 kHz Resampling and Log-Mel Feature Extraction

The `WhisperFeatureExtractor` converts a raw 16 kHz waveform into an 80-bin log-Mel spectrogram of fixed size `[80, 3000]` (representing 30 seconds). The `WhisperTokenizer` converts a transcript string into token IDs. Both live inside `WhisperProcessor`.

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
import numpy as np

WHISPER_MODEL_ID = "openai/whisper-small"
TARGET_SR = 16000

# --- Feature extractor (audio side) ---
feature_extractor = WhisperFeatureExtractor.from_pretrained(WHISPER_MODEL_ID)

print("WhisperFeatureExtractor config:")
print(f"  sampling_rate : {feature_extractor.sampling_rate} Hz")
print(f"  n_mels        : {feature_extractor.feature_size}")
print(f"  hop_length    : {feature_extractor.hop_length} samples ({feature_extractor.hop_length/TARGET_SR*1000:.0f}ms)")
print(f"  n_fft         : {feature_extractor.n_fft} samples ({feature_extractor.n_fft/TARGET_SR*1000:.0f}ms)")
print(f"  chunk_length  : {feature_extractor.chunk_length}s  -> {feature_extractor.nb_max_frames} frames")

# --- Tokenizer (text side) ---
# Set language="french" and task="transcribe" so the correct special tokens
# are prepended to every label sequence.
tokenizer = WhisperTokenizer.from_pretrained(
    WHISPER_MODEL_ID,
    language="french",
    task="transcribe",
)

print("\nWhisperTokenizer:")
print(f"  vocab_size  : {tokenizer.vocab_size}")
print(f"  bos_token   : {repr(tokenizer.bos_token)}  (id={tokenizer.bos_token_id})")
print(f"  eos_token   : {repr(tokenizer.eos_token)}  (id={tokenizer.eos_token_id})")
print(f"  pad_token   : {repr(tokenizer.pad_token)}  (id={tokenizer.pad_token_id})")

# Demonstrate tokenization
sample_sentence = "Bonjour, je suis en train d'apprendre le francais."
tokens = tokenizer(sample_sentence)
decoded = tokenizer.decode(tokens["input_ids"], skip_special_tokens=True)
print(f"\nSample: {repr(sample_sentence)}")
print(f"  Token IDs : {tokens['input_ids']}")
print(f"  Pieces    : {[tokenizer.decode([t]) for t in tokens['input_ids']]}")
print(f"  Decoded   : {repr(decoded)}")

# Full processor wraps both
processor_fr = WhisperProcessor.from_pretrained(
    WHISPER_MODEL_ID,
    language="french",
    task="transcribe",
)
print("\nWhisperProcessor ready.")

# --- Audio preprocessing demo ---
# Cast to 16kHz (datasets handles resampling on load when the Audio feature is set)
from datasets import Audio as HFAudio
cv_fr_16k = cv_fr.cast_column("audio", HFAudio(sampling_rate=TARGET_SR))

raw = cv_fr_16k[0]
waveform = np.array(raw["audio"]["array"], dtype=np.float32)
features = feature_extractor(waveform, sampling_rate=TARGET_SR, return_tensors="np")
print(f"\nAudio waveform shape : {waveform.shape}")
print(f"input_features shape : {features.input_features.shape}")   # [1, 80, 3000]

In [ ]:
TARGET_SR = 16000

# Cast the audio column to 16kHz. The datasets library resamples on the fly.
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=TARGET_SR))

# Keep only what we need
KEEP_COLS = {"audio", "sentence"}  # audio = waveform, sentence = transcript
cols_to_remove = [c for c in common_voice["train"].column_names if c not in KEEP_COLS]
common_voice = common_voice.remove_columns(cols_to_remove)

print(common_voice)

# Verify resampling worked
sample = common_voice["train"][0]
print(f"\nAudio sample rate: {sample['audio']['sampling_rate']} Hz")
print(f"Audio shape: {np.array(sample['audio']['array']).shape}")
print(f"Transcript: {repr(sample['sentence'])}")

### Resampling Without the `datasets` Library

If you're not using `datasets` and need to resample raw audio:

In [ ]:
# Option 1: librosa (works on numpy arrays)
def resample_with_librosa(audio_array: np.ndarray, orig_sr: int, target_sr: int = 16000) -> np.ndarray:
    if orig_sr == target_sr:
        return audio_array
    return librosa.resample(audio_array, orig_sr=orig_sr, target_sr=target_sr)


# Option 2: torchaudio (works on tensors, faster on GPU)
def resample_with_torchaudio(waveform: torch.Tensor, orig_sr: int, target_sr: int = 16000) -> torch.Tensor:
    if orig_sr == target_sr:
        return waveform
    resampler = T.Resample(orig_freq=orig_sr, new_freq=target_sr)
    return resampler(waveform)


# Demo: resample a numpy array from 48kHz to 16kHz
fake_48k = np.random.randn(48000).astype(np.float32)  # 1 second at 48kHz
resampled = resample_with_librosa(fake_48k, orig_sr=48000, target_sr=16000)
print(f"48kHz input: {len(fake_48k)} samples -> 16kHz output: {len(resampled)} samples")
print(f"Ratio: {len(fake_48k) / len(resampled):.1f}x")

## 2. Tokenization for ASR: WhisperProcessor

The `WhisperProcessor` does double duty:

1. **Feature extraction** (audio side): converts a raw waveform into a log-Mel spectrogram (`input_features`)
2. **Tokenization** (text side): converts transcript strings into token IDs (`labels`)

Both are needed to set up supervised training. The feature extractor handles audio; the tokenizer handles text. They share a single `WhisperProcessor` object.

We configure the processor with the target language and task so the correct special tokens are prepended to the decoder.

In [ ]:
processor = WhisperProcessor.from_pretrained(
    MODEL_ID,
    language=LANGUAGE_NAME,
    task="transcribe",
)

# Inspect what the tokenizer looks like
sample_text = "Sut ydych chi?"
tokens = processor.tokenizer(sample_text)
print(f"Text: {repr(sample_text)}")
print(f"Token IDs: {tokens['input_ids']}")
print(f"Decoded back: {repr(processor.tokenizer.decode(tokens['input_ids']))}")

# The tokenizer uses BPE - subword pieces, not whole words
pieces = [processor.tokenizer.decode([tid]) for tid in tokens['input_ids']]
print(f"Token pieces: {pieces}")

In [ ]:
def preprocess_example(example: dict) -> dict:
    """
    Convert a dataset example into model inputs.
    
    Input dict keys: 'audio' (dict with 'array' and 'sampling_rate'), 'sentence'
    Output dict keys: 'input_features', 'labels'
    """
    audio = example["audio"]
    waveform = np.array(audio["array"], dtype=np.float32)

    # Feature extraction: waveform -> log-Mel spectrogram
    # processor.feature_extractor returns a dict with 'input_features'
    features = processor.feature_extractor(
        waveform,
        sampling_rate=TARGET_SR,
        return_tensors="np",
    )
    input_features = features.input_features[0]  # shape: [80, 3000]

    # Tokenize the transcript
    # We set the padding token to -100 so the loss ignores padding positions
    labels = processor.tokenizer(
        example["sentence"],
        return_tensors="np",
    ).input_ids[0]

    return {
        "input_features": input_features,
        "labels": labels,
    }


# Test the preprocessing function
sample = common_voice["train"][0]
processed = preprocess_example(sample)
print("input_features shape:", processed["input_features"].shape)  # [80, 3000]
print("labels:", processed["labels"])
print("decoded label:", processor.tokenizer.decode(processed["labels"], skip_special_tokens=True))

In [ ]:
# Apply preprocessing to the entire dataset
# num_proc=1 for safety on first run; increase for speed if your system allows
common_voice = common_voice.map(
    preprocess_example,
    remove_columns=common_voice["train"].column_names,
    num_proc=1,
    desc="Preprocessing audio",
)

print(common_voice)
print("Features:", common_voice["train"].features)

## 3. Data Augmentation

Audio augmentation makes the model more robust to real-world variation. Three techniques are standard in ASR:

### SpecAugment
SpecAugment operates on the spectrogram (after feature extraction). It randomly masks:
- **Frequency bands** (horizontal stripes in the spectrogram)
- **Time steps** (vertical stripes)

This forces the model to use context rather than relying on any single frequency band or short time window. It was introduced specifically for ASR and gives consistent improvements.

### Noise Injection (brief note)
Add Gaussian noise to the raw waveform: `audio + noise_scale * np.random.randn(*audio.shape)`. This helps with recordings in noisy environments. Typical `noise_scale` values: 0.001 to 0.005.

### Speed Perturbation (brief note)
Stretch or compress the audio in time by a small factor (0.9x to 1.1x) using `librosa.effects.time_stretch`. This makes the model robust to speaking rate variation. Apply **before** feature extraction.

In [ ]:
class SpecAugment(torch.nn.Module):
    """
    SpecAugment: frequency and time masking on a log-Mel spectrogram.
    Input shape: [batch, n_mels, time] or [n_mels, time]
    """

    def __init__(
        self,
        freq_mask_param: int = 27,    # max width of frequency mask (in mel bins)
        time_mask_param: int = 100,   # max width of time mask (in frames)
        n_freq_masks: int = 2,
        n_time_masks: int = 2,
    ):
        super().__init__()
        self.freq_masks = torch.nn.ModuleList([
            T.FrequencyMasking(freq_mask_param=freq_mask_param)
            for _ in range(n_freq_masks)
        ])
        self.time_masks = torch.nn.ModuleList([
            T.TimeMasking(time_mask_param=time_mask_param)
            for _ in range(n_time_masks)
        ])

    def forward(self, spectrogram: torch.Tensor) -> torch.Tensor:
        # torchaudio's masking transforms expect shape [channel, freq, time]
        if spectrogram.ndim == 2:
            spectrogram = spectrogram.unsqueeze(0)  # add channel dim
            squeeze_back = True
        else:
            squeeze_back = False

        for mask in self.freq_masks:
            spectrogram = mask(spectrogram)
        for mask in self.time_masks:
            spectrogram = mask(spectrogram)

        if squeeze_back:
            spectrogram = spectrogram.squeeze(0)
        return spectrogram


spec_augment = SpecAugment(freq_mask_param=27, time_mask_param=100, n_freq_masks=2, n_time_masks=2)

# Demo on a fake spectrogram
fake_spec = torch.ones(80, 3000)  # [n_mels, time_frames]
augmented = spec_augment(fake_spec)
n_masked = (augmented == 0).sum().item()
print(f"Original spec: all ones ({fake_spec.numel()} values)")
print(f"After SpecAugment: {n_masked} values masked to 0 ({100*n_masked/fake_spec.numel():.1f}%)")

In [ ]:
def add_gaussian_noise(audio: np.ndarray, noise_scale: float = 0.002) -> np.ndarray:
    """Add Gaussian noise to a waveform. Apply BEFORE feature extraction."""
    noise = noise_scale * np.random.randn(*audio.shape).astype(np.float32)
    return np.clip(audio + noise, -1.0, 1.0)


def speed_perturb(audio: np.ndarray, rate: float = None) -> np.ndarray:
    """
    Randomly stretch or compress audio in time.
    rate < 1.0 = slower (more frames), rate > 1.0 = faster (fewer frames).
    If rate is None, sample uniformly from [0.9, 1.1].
    Apply BEFORE feature extraction.
    """
    if rate is None:
        rate = np.random.uniform(0.9, 1.1)
    return librosa.effects.time_stretch(audio, rate=rate)


# Demo
fake_audio = np.random.randn(16000).astype(np.float32)  # 1 second
noisy = add_gaussian_noise(fake_audio, noise_scale=0.002)
sped_up = speed_perturb(fake_audio, rate=1.1)
slowed = speed_perturb(fake_audio, rate=0.9)

print(f"Original:    {len(fake_audio)} samples")
print(f"Noisy:       {len(noisy)} samples (same length, different values)")
print(f"Sped up 1.1x: {len(sped_up)} samples")
print(f"Slowed 0.9x:  {len(slowed)} samples")

## 4. Data Collator

A data collator's job is to take a list of individual examples from the dataset and combine them into a batch. For Whisper fine-tuning, the collator needs to:

1. Stack `input_features` tensors (these are all the same shape: `[80, 3000]`)
2. Pad `labels` to the same length within the batch (transcripts have variable lengths)
3. Replace padding positions in labels with `-100` (the value PyTorch's cross-entropy loss ignores)

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # --- Input features ---
        # All input_features are already [80, 3000], so we just stack them.
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        # --- Labels (transcripts) ---
        # These are variable-length token ID sequences.
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        # Replace padding token ID with -100 so the loss ignores it.
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Strip the BOS (beginning of sequence) token from the front of each label.
        # The model generates this token itself; training with it in the labels
        # causes the model to shift the sequence.
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# Test the collator with 2 examples
sample_batch = [common_voice["train"][0], common_voice["train"][1]]
collated = data_collator(sample_batch)
print("Collated batch keys:", list(collated.keys()))
print("input_features:", collated["input_features"].shape)  # [2, 80, 3000]
print("labels:", collated["labels"].shape)                  # [2, max_label_len]

## 5. Evaluation Metric

The standard metric for ASR is Word Error Rate (WER): the fraction of words that are wrong. Lower is better. We'll set up the metric function the `Trainer` will call after each evaluation step.

In [ ]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    """Called by Seq2SeqTrainer at the end of each evaluation epoch."""
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 with the pad token ID for decoding
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 4)}

print("compute_metrics function ready.")

## 6. Fine-Tuning with Seq2SeqTrainer

The `Seq2SeqTrainer` extends HuggingFace's standard `Trainer` with two things specific to encoder-decoder models:
1. `predict_with_generate=True`: at evaluation time, use `model.generate()` instead of teacher-forced forward passes. This gives a more realistic WER.
2. `generation_max_length`: caps the maximum length of generated sequences.

### Key Training Arguments

| Argument | What it controls |
|---|---|
| `per_device_train_batch_size` | Batch size per GPU/CPU |
| `gradient_accumulation_steps` | Accumulate gradients over N steps before updating; effective batch = batch_size × accumulation |
| `learning_rate` | Starting LR. Whisper fine-tuning typically uses 1e-5 to 5e-5 |
| `warmup_steps` | Number of steps to linearly ramp up LR from 0 |
| `max_steps` | Total training steps. Use this to control run length for quick experiments |
| `fp16` | Enable 16-bit training (requires CUDA). Cuts memory roughly in half |
| `eval_steps` | Run evaluation every N steps |
| `save_steps` | Save a checkpoint every N steps |

In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# Configure the model to always use our target language and task
model.generation_config.language = LANGUAGE_NAME
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None  # let processor handle this

# For training efficiency, disable the cache (not needed during training)
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params/1e6:.1f}M")
print(f"Trainable parameters: {trainable_params/1e6:.1f}M")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-finetuned-cy",

    # --- Compute ---
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,  # increase to 2 or 4 if you get OOM
    fp16=torch.cuda.is_available(),  # enable if on GPU
    dataloader_num_workers=0,        # set to 4+ for faster data loading

    # --- Optimization ---
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=500,           # set to a small number for a quick demo run
    weight_decay=0.01,

    # --- Logging and checkpointing ---
    logging_steps=25,
    eval_steps=100,
    save_steps=100,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,         # lower WER is better

    # --- Seq2Seq specific ---
    predict_with_generate=True,
    generation_max_length=225,

    report_to="none",  # set to "wandb" if you have W&B configured
)

print("Training arguments configured.")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=common_voice["train"],
    eval_dataset=common_voice["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,  # needed for checkpoint saving
)

print("Trainer ready.")
print(f"Train examples: {len(common_voice['train'])}")
print(f"Eval examples:  {len(common_voice['test'])}")

## 6b. Smoke Test: 10-Step Pipeline Verification

Before committing to a full training run, do a 10-step smoke test. This verifies that the data pipeline, collator, model forward pass, loss, and metric computation all work end to end without errors.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# Smoke-test training arguments: only 10 steps, no checkpointing
smoke_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-smoke-test",
    max_steps=10,                        # stop after 10 optimizer steps
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    fp16=torch.cuda.is_available(),
    learning_rate=1e-5,
    warmup_steps=0,
    logging_steps=1,                     # log every step so we can watch the loss
    eval_steps=5,
    evaluation_strategy="steps",
    save_strategy="no",                  # no checkpoints for the smoke test
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    dataloader_num_workers=0,
)

# Take a tiny slice of the data so each step is fast
train_small = common_voice["train"].select(range(20))
eval_small  = common_voice["test"].select(range(10))

# Fresh model for the smoke test
smoke_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
smoke_model.config.use_cache = False
smoke_model.generation_config.language = LANGUAGE_NAME
smoke_model.generation_config.task = "transcribe"
smoke_model.generation_config.forced_decoder_ids = None

smoke_trainer = Seq2SeqTrainer(
    model=smoke_model,
    args=smoke_args,
    train_dataset=train_small,
    eval_dataset=eval_small,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print("Starting 10-step smoke test ...")
smoke_trainer.train()
print("\nSmoke test passed. The pipeline is working end to end.")
print("Check that loss values printed above are finite (not NaN or Inf).")

## 6c. Reducing VRAM: Gradient Checkpointing and Frozen Feature Encoder

Two techniques that significantly reduce GPU memory usage during fine-tuning:

**Gradient checkpointing** trades compute for memory. Instead of keeping all intermediate activations in memory for the backward pass, it recomputes them on demand. This roughly halves the activation memory at the cost of about 20% more compute per step.

**Freezing the feature encoder** stops gradients from flowing through Whisper's CNN audio frontend. The CNN has already learned a strong audio representation during pre-training and rarely needs to be updated for fine-tuning. Freezing it saves both memory and compute.

In [ ]:
from transformers import WhisperForConditionalGeneration

# Load a fresh model copy to demonstrate VRAM reduction techniques
vram_model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

total_params = sum(p.numel() for p in vram_model.parameters())
trainable_before = sum(p.numel() for p in vram_model.parameters() if p.requires_grad)
print(f"Before freezing:")
print(f"  Total params     : {total_params/1e6:.1f}M")
print(f"  Trainable params : {trainable_before/1e6:.1f}M")

# --- 1. Freeze the CNN feature encoder ---
# This sets requires_grad=False on all parameters of model.model.encoder.conv1
# and model.model.encoder.conv2 (the two convolutional layers before the
# transformer encoder). Whisper exposes this as a convenience method.
vram_model.freeze_feature_encoder()

trainable_after = sum(p.numel() for p in vram_model.parameters() if p.requires_grad)
frozen = trainable_before - trainable_after
print(f"\nAfter freeze_feature_encoder():")
print(f"  Trainable params : {trainable_after/1e6:.1f}M  (-{frozen/1e6:.1f}M frozen)")

# --- 2. Enable gradient checkpointing ---
# Must be called AFTER moving the model to device and BEFORE wrapping in Trainer.
# The Trainer also accepts gradient_checkpointing=True in training args,
# which calls this automatically, but calling it manually is clearer.
vram_model.gradient_checkpointing_enable()
# Disable the KV cache (incompatible with gradient checkpointing during training)
vram_model.config.use_cache = False
print("\nGradient checkpointing enabled.")
print("use_cache set to False (required for gradient checkpointing).")

# Show which modules are frozen
frozen_modules = [
    name for name, param in vram_model.named_parameters() if not param.requires_grad
]
print(f"\nFrozen parameters ({len(frozen_modules)} tensors):")
for n in frozen_modules[:8]:
    print(f"  {n}")
if len(frozen_modules) > 8:
    print(f"  ... and {len(frozen_modules) - 8} more")

In [ ]:
# Run training
# On a GPU this will take ~5-10 minutes for 500 steps with whisper-tiny.
# On CPU it will be much slower; reduce max_steps to 50 for a quick sanity check.
trainer.train()

In [ ]:
# Save the final model and processor together
# This creates a directory with all files needed for model.from_pretrained() / pipeline()
trainer.save_model()
processor.save_pretrained(training_args.output_dir)
print(f"Model saved to: {training_args.output_dir}")

### Loading and Using the Fine-Tuned Model

In [ ]:
from transformers import pipeline as hf_pipeline

# Load your fine-tuned model exactly like you'd load any HuggingFace model
finetuned_asr = hf_pipeline(
    task="automatic-speech-recognition",
    model=training_args.output_dir,
    device=0 if torch.cuda.is_available() else -1,
)

# Transcribe a validation example
test_example = common_voice["test"][0]
# Reconstruct the audio from the preprocessed features is non-trivial;
# easier to re-load from the original dataset before preprocessing.
# (If you still have the original audio file path, pass it directly.)
print("Fine-tuned model loaded successfully.")
print("Use finetuned_asr('path/to/audio.wav') to transcribe new files.")

## Exercise

1. **Switch languages**: Replace `LANGUAGE = "cy"` with another language from Common Voice (e.g., `"ta"` for Tamil, `"sw"` for Swahili, `"eu"` for Basque). Update `LANGUAGE_NAME` accordingly. Re-run the full pipeline. Does the base Whisper-tiny model perform better or worse before fine-tuning for your chosen language?

2. **SpecAugment ablation**: Run two short training jobs (50 steps each): one with SpecAugment applied to `input_features` in the data collator, one without. Compare the eval WER. To add SpecAugment in the collator, apply `spec_augment(batch['input_features'])` before returning from `__call__`.

3. **Gradient accumulation**: The effective batch size matters for Whisper fine-tuning. Try `per_device_train_batch_size=4, gradient_accumulation_steps=4` (effective batch=16) vs. `per_device_train_batch_size=16, gradient_accumulation_steps=1`. Are results similar? When would you choose accumulation over a larger batch size?

4. **(Stretch)** Use PEFT/LoRA instead of full fine-tuning. Install `peft`, wrap the model with `get_peft_model(model, LoraConfig(r=8, target_modules=["q_proj", "v_proj"]))`, and run the same training script. How many trainable parameters does LoRA use vs. full fine-tuning? Does the WER differ much at the end?

## Exercise: Extend compute_metrics to Report CER

The `compute_metrics` function currently only reports WER. Modify it to also compute and return CER (Character Error Rate). CER is often more informative for morphologically rich languages where a single wrong character inflates WER unfairly.

In [ ]:
# pip install evaluate jiwer  # uncomment if needed
import evaluate

# YOUR CODE HERE
# Steps:
#   1. Load the CER metric alongside WER:
#        cer_metric = evaluate.load("cer")
#
#   2. Write a new function `compute_metrics_wer_cer(pred)` that:
#        a. Replaces -100 in label_ids with pad_token_id (same as existing compute_metrics)
#        b. Decodes predictions and labels with skip_special_tokens=True
#        c. Computes WER using wer_metric.compute(predictions=..., references=...)
#        d. Computes CER using cer_metric.compute(predictions=..., references=...)
#        e. Returns {"wer": ..., "cer": ...}
#
#   3. Verify your function manually with these example inputs:
#        pred_strings = ["bonjour le monde", "je suis content"]
#        ref_strings  = ["bonjour le monde", "je suis content"]
#        # Expected: wer=0.0, cer=0.0
#
#        pred_strings = ["bonjur le monde", "je suis contente"]
#        ref_strings  = ["bonjour le monde", "je suis content"]
#        # Expected: some non-zero WER and CER

raise NotImplementedError